# CIFAR-100 RAPS with Class Similarity – Demo

This notebook demonstrates how to run all experiment variants using the
refactored `conformal` package.

### Module layout

```
conformal/
  config.py                 # global hyperparameters (edit here)
  data_loader.py            # load logits, adjacency & similarity matrices
  score_functions.py        # RAPS score computation + prediction-set builders
  metrics.py                # coverage / size evaluation helpers
  experiments_original.py   # paper-replication runners (not exchangeability-safe)
  experiments_exchangability.py  # exchangability runners (exchangeability-safe)
```

### Replication vs exchangability

Functions in `experiments_original.py` mirror the original paper's protocol:
when λ is selected via a validation fold, that fold is **merged back** into
calibration before computing q̂.  This violates the exchangeability assumption
of conformal prediction.

Functions in `experiments_exchangability.py` fix this by keeping calibration,
validation (λ selection), and test strictly separate.

In [1]:
import sys
sys.path.insert(0, '..')   # so 'conformal' is importable

%load_ext autoreload
%autoreload 2

import numpy as np
import src.config as config
import src.metrics as metrics
import src.score_functions as score_functions
import src.experiments_exchangability as experiments_exchangability
import src.data_loader as data_loader
import src.experiments_original as experiments_original




## 1. Load data

In [3]:
# Edit these paths to point at your .npz files
LOGITS_PATH      = '../data/Cifar100_ResNet50_logits_new.npz'
ADJACENCY_PATH   = '../data/Adjacency_matrix.npz'
SIMILARITY_PATH  = '../data/similarity_matrix_cosine_resnet50.npz'

probabilities, labels = data_loader.load_logits(LOGITS_PATH)
adjacency_matrix, adjacency_matrix_smaller = data_loader.load_adjacency_matrix(ADJACENCY_PATH)
adjacency_matrix_ms = data_loader.load_similarity_matrix_cosine(SIMILARITY_PATH)

print('probabilities:', probabilities.shape)
print('labels:       ', labels.shape)
print('adjacency:    ', adjacency_matrix.shape)

probabilities: (10000, 100)
labels:        (10000,)
adjacency:     (100, 101)


## 2. Baseline RAPS (no class-similarity penalty)

In [ ]:
results_baseline = run_raps_baseline(probabilities, labels)

## 3. RAPS + MA superclass-mass penalty  (original protocol)

In [ ]:
lamda_2_values = [0, 0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.3]

results_ma_orig = conformal.run_raps_ma_avg_opt(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_2_values,
)

## 4. RAPS + MA superclass-mass penalty  (exchangability – exchangeability-safe)

In [ ]:
results_ma_corr = conformal.run_raps_ma_avg_opt_exchangability(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_2_values,
)

## 5. RAPS + MS binary-distance penalty  (exchangability)

In [ ]:
results_ms_corr = conformal.run_raps_ms_avg_opt_exchangability(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_ms,
    lamda_2_values,
)

## 6. Compare results

In [ ]:
import pandas as pd

rows = {}
for name, r in [
    ('Baseline',    results_baseline),
    ('MA original', results_ma_orig),
    ('MA exchangability', results_ma_corr),
    ('MS exchangability', results_ms_corr),
]:
    rows[name] = {
        'Coverage':         f"{r['marginal_coverage']:.4f}",
        'Mean CC':          f"{r['mean_class_cc']:.4f}",
        'Worst CC dev':     f"{r['worst_cc_deviation']:.4f}",
        'Top-5 CC dev':     f"{r['top5_cc_deviation']:.4f}",
        'Avg set size':     f"{r['set_sizes']['mean']:.3f}",
    }

pd.DataFrame(rows).T